# Livrable 1 — Modélisation formelle du problème
## Optimisation de tournées avec contraintes

**Structure :** CesiCDP  
**Équipe :** Groupe 1

---

> *Ce notebook constitue le premier livrable du projet. Il présente le contexte, reformule le problème de manière formelle, intègre les contraintes retenues par l'équipe, et en étudie les propriétés théoriques de complexité. Les méthodes de résolution ne sont pas abordées ici.*

## Table des matières

1. [Contexte et motivation](#1)
2. [Définition informelle du problème](#2)
3. [Représentation formelle des données](#3)
4. [Formulation du problème d'optimisation](#4)
5. [Contraintes supplémentaires retenues](#5)
   - 5.1 [Fenêtres temporelles (Time Windows)](#5-1)
   - 5.2 [Coût et restrictions sur certaines arêtes](#5-2)
6. [Formulation complète du modèle (TSPTW-R)](#6)
7. [Analyse de la complexité théorique](#7)
8. [Références bibliographiques](#8)

---
## 1. Contexte et motivation <a id="1"></a>

### 1.1 Enjeux environnementaux et réglementaires

Depuis la signature du **Protocole de Kyoto (1997)** et l'Accord de Paris (2015), les États et collectivités s'engagent à réduire drastiquement leurs émissions de gaz à effet de serre. En France, l'objectif est une division par quatre des émissions d'ici 2050 (facteur 4). Le secteur des transports représente à lui seul **près de 30% des émissions de CO₂** en France (ADEME, 2023), ce qui en fait un levier d'action prioritaire.

### 1.2 L'appel à manifestation d'intérêt de l'ADEME

L'**ADEME** (Agence de l'Environnement et de la Maîtrise de l'Énergie) a lancé un appel à manifestation d'intérêt pour financer des **démonstrateurs de nouvelles solutions de mobilité** pour personnes et marchandises. L'objectif est d'identifier des solutions algorithmiques capables de :

- Réduire le nombre de kilomètres parcourus par les véhicules de livraison
- Minimiser la consommation énergétique et les émissions associées
- S'adapter aux contraintes opérationnelles réelles (horaires, capacités, contraintes routières)

### 1.3 Positionnement de CesiCDP

La structure **CesiCDP**, déjà implantée dans le domaine de la mobilité multimodale intelligente, répond à cet appel. Les applications visées sont multiples :

| Application | Description |
|---|---|
| Distribution du courrier | Tournées de facteurs avec contraintes horaires |
| Livraison de colis | Respect de créneaux client, restrictions de zones |
| Collecte de déchets | Optimisation des circuits de ramassage |
| Traitement routier | Salage, entretien selon priorités et fenêtres météo |

### 1.4 Objectif algorithmique

Le problème central consiste à **calculer sur un réseau routier une tournée optimale** reliant un sous-ensemble de villes (ou points de livraison), en revenant au point de départ, de manière à **minimiser la durée totale du trajet**, tout en respectant les contraintes opérationnelles réelles.

---
## 2. Définition informelle du problème <a id="2"></a>

### 2.1 Le problème du Voyageur de Commerce (TSP)

Le problème de base sous-jacent est le célèbre **Travelling Salesman Problem (TSP)**, formulé historiquement par Dantzig, Fulkerson et Johnson en 1954 [1]. Dans sa version classique :

> *Étant donné un ensemble de villes et les distances entre chacune d'elles, trouver le chemin le plus court passant par toutes les villes exactement une fois et revenant à la ville de départ.*

### 2.2 Extension à notre contexte opérationnel

Notre problème s'écarte du TSP classique sur plusieurs points essentiels :

1. **Le graphe n'est pas nécessairement complet** : certaines routes n'existent pas ou sont interdites (travaux, zones à accès restreint).
2. **Les arêtes ont des coûts hétérogènes** : certaines routes sont plus coûteuses (péages, voies dégradées, zones de congestion).
3. **Les villes ont des fenêtres temporelles** : un point de livraison n'est accessible que dans un créneau horaire `[a_i, b_i]`.

Ce problème enrichi est connu dans la littérature sous le nom de **TSP with Time Windows and Edge Restrictions (TSPTW-R)**.

### 2.3 Analogie intuitive

Imaginons un livreur qui doit déposer des colis dans plusieurs quartiers d'une ville. Certains clients ne sont disponibles qu'entre 9h et 11h (fenêtres temporelles). Certaines rues sont en travaux et donc interdites ou plus longues à parcourir (restrictions d'arêtes). Le livreur cherche l'ordre de visite qui minimise son temps total de tournée tout en honorant tous les créneaux.

---
## 3. Représentation formelle des données <a id="3"></a>

### 3.1 Structure de graphe

Le réseau routier est modélisé par un **graphe orienté pondéré** :

$$G = (V, E, w, c)$$

où :

| Symbole | Type | Définition |
|---|---|---|
| $V$ | Ensemble fini | Ensemble des $n$ sommets (villes / points de livraison) |
| $E \subseteq V \times V$ | Ensemble d'arêtes | Routes existantes entre les villes |
| $w : E \rightarrow \mathbb{R}^+$ | Fonction de coût | Temps de parcours (ou distance) de chaque arête |
| $c : E \rightarrow \{0, 1, \mathbb{R}^+\}$ | Fonction de restriction | $0$ = arête interdite, $r > 1$ = surcoût multiplicatif |

**Convention :** Le sommet $v_0 \in V$ désigne le **dépôt** (point de départ et d'arrivée de la tournée).

### 3.2 Données associées aux sommets

À chaque sommet $v_i \in V \setminus \{v_0\}$ est associé un vecteur d'attributs :

$$v_i \leftarrow (a_i, b_i, s_i)$$

où :
- $a_i \in \mathbb{R}^+$ : **début de la fenêtre temporelle** (earliest arrival time)
- $b_i \in \mathbb{R}^+$ : **fin de la fenêtre temporelle** (latest arrival time), avec $a_i \leq b_i$
- $s_i \in \mathbb{R}^+$ : **temps de service** au sommet $i$ (durée de la livraison)

Le dépôt $v_0$ a par convention $a_0 = 0$, $b_0 = +\infty$, $s_0 = 0$.

### 3.3 Données associées aux arêtes

Le **coût effectif** de traversée d'une arête $(i, j) \in E$ est défini par :

$$\tilde{w}(i,j) = 
\begin{cases}
+\infty & \text{si } c(i,j) = 0 \quad \text{(arête interdite)} \\
w(i,j) \times c(i,j) & \text{si } c(i,j) > 0 \quad \text{(coût normal ou majoré)}
\end{cases}$$

Ainsi, les arêtes interdites sont effectivement exclues de toute solution faisable, et les arêtes coûteuses sont pénalisées proportionnellement.

### 3.4 Représentation matricielle

Pour faciliter l'implémentation algorithmique, on représente le graphe par deux matrices $n \times n$ :

- **Matrice de temps de parcours** $W$ : $W[i][j] = w(i,j)$ si $(i,j) \in E$, $+\infty$ sinon
- **Matrice de restrictions** $C$ : $C[i][j] = c(i,j)$ si $(i,j) \in E$, $0$ sinon
- **Matrice de coûts effectifs** $\tilde{W}$ : $\tilde{W}[i][j] = \tilde{w}(i,j)$

---
## 4. Formulation du problème d'optimisation <a id="4"></a>

### 4.1 Variables de décision

On introduit les variables binaires :

$$x_{ij} \in \{0, 1\}, \quad \forall (i,j) \in E$$

avec la sémantique : $x_{ij} = 1$ si et seulement si l'arête $(i, j)$ est **empruntée** dans la tournée.

On définit également les variables de temps de passage :

$$t_i \in \mathbb{R}^+, \quad \forall i \in V$$

représentant le **temps d'arrivée** au sommet $i$.

### 4.2 Fonction objectif

On cherche à **minimiser le coût total de la tournée** :

$$\min \sum_{(i,j) \in E} \tilde{w}(i,j) \cdot x_{ij}$$

### 4.3 Contraintes de base (TSP)

**(C1) — Chaque sommet est visité exactement une fois :**

$$\sum_{j : (i,j) \in E} x_{ij} = 1, \quad \forall i \in V$$

$$\sum_{i : (i,j) \in E} x_{ij} = 1, \quad \forall j \in V$$

**(C2) — Élimination des sous-tours (contraintes de Miller-Tucker-Zemlin [2]) :**

On introduit des variables auxiliaires $u_i \in \{1, \ldots, n\}$ :

$$u_i - u_j + n \cdot x_{ij} \leq n - 1, \quad \forall i \neq j \in V \setminus \{v_0\}$$

*Note : Ces contraintes MTZ garantissent que la solution forme un unique cycle hamiltonien, sans sous-tours disjoints.*

**(C3) — Intégrité des variables :**

$$x_{ij} \in \{0, 1\}, \quad u_i \in \mathbb{Z}^+$$

---
## 5. Contraintes supplémentaires retenues <a id="5"></a>

Conformément aux exigences du projet, l'équipe a retenu **deux contraintes supplémentaires** parmi la liste proposée. Ces contraintes ont été choisies pour leur pertinence applicative (adéquation avec les cas réels de livraison) et leur relative accessibilité d'implémentation.

---

### 5.1 Contrainte 1 — Fenêtres temporelles (Time Windows) <a id="5-1"></a>

#### Motivation

Dans la réalité, les destinataires d'une livraison ne sont pas disponibles à toute heure. Un commerce ouvre à 8h et ferme à 18h, un particulier est disponible uniquement entre 14h et 16h. Le livreur doit **arriver dans le créneau autorisé**, sous peine de devoir repartir sans effectuer la livraison.

#### Formalisation

Chaque sommet $v_i \in V \setminus \{v_0\}$ est associé à un intervalle $[a_i, b_i]$. La **variable de temps de passage** $t_i$ doit satisfaire :

$$a_i \leq t_i \leq b_i, \quad \forall i \in V \setminus \{v_0\}$$

#### Cohérence temporelle entre sommets successifs

Si le sommet $j$ est visité juste après le sommet $i$ (i.e. $x_{ij} = 1$), alors le temps d'arrivée en $j$ doit respecter :

$$t_j \geq (t_i + s_i + \tilde{w}(i,j)) \cdot x_{ij}, \quad \forall (i,j) \in E$$

Pour linéariser cette contrainte, on utilise la forme MTZ avec big-M [3] :

$$t_j \geq t_i + s_i + \tilde{w}(i,j) - M(1 - x_{ij}), \quad \forall (i,j) \in E$$

avec $M$ une constante suffisamment grande (typiquement $M = \sum_{(i,j) \in E} \tilde{w}(i,j)$).

#### Gestion des attentes

Si le livreur arrive avant l'ouverture de la fenêtre ($t_i < a_i$), il **attend** sur place. Le temps effectif de départ de $i$ est donc $\max(t_i, a_i) + s_i$. Cela se modélise en redéfinissant le temps de départ effectif :

$$d_i = \max(t_i, a_i) + s_i$$

#### Type de fenêtres : hard vs soft

| Type | Définition | Notre choix |
|---|---|---|
| **Hard Time Windows** | $t_i > b_i$ rend la solution **infaisable** | ✓ Retenu |
| Soft Time Windows | $t_i > b_i$ est autorisé mais **pénalisé** dans l'objectif | Non retenu |

Nous utilisons des **fenêtres dures** (hard), ce qui est la formulation la plus répandue dans la littérature TSPTW [4].

---

### 5.2 Contrainte 2 — Coût et restrictions sur certaines arêtes <a id="5-2"></a>

#### Motivation

Un réseau routier réel n'est pas homogène. Certains axes sont temporairement fermés (travaux, accidents), d'autres sont soumis à des péages ou présentent des difficultés de circulation (zones piétonnes partielles, gabarit limité). Modéliser ces hétérogénéités est essentiel pour obtenir des tournées réalistes.

#### Formalisation

On enrichit le graphe avec la fonction de restriction $c : E \rightarrow \mathbb{R}^+ \cup \{0\}$ définie comme :

$$c(i,j) = \begin{cases}
0 & \text{route interdite (ex : travaux, sens interdit)} \\
1 & \text{route normale} \\
r > 1 & \text{surcoût multiplicatif (ex : péage, voie dégradée)}
\end{cases}$$

Le **coût effectif** de l'arête $(i,j)$ devient :

$$\tilde{w}(i,j) = \begin{cases}
+\infty & \text{si } c(i,j) = 0 \\
w(i,j) \times c(i,j) & \text{sinon}
\end{cases}$$

#### Impact sur la faisabilité

La présence d'arêtes interdites peut rendre le graphe **non-hamiltonien** : il peut exister des instances où aucune tournée complète n'est réalisable. Il faut donc, lors de la génération d'instances, s'assurer de la **connexité** du graphe résiduel (après suppression des arêtes interdites).

**Condition nécessaire de faisabilité :**

$$\forall i \in V, \quad \exists j \in V : (i,j) \in E \text{ et } c(i,j) > 0$$

#### Contrainte d'exclusion dans le modèle ILP

Les arêtes interdites sont simplement exclues de l'ensemble $E$ des arêtes admissibles :

$$E' = \{(i,j) \in E \mid c(i,j) > 0\}$$

$$x_{ij} = 0, \quad \forall (i,j) \in E \setminus E'$$

Les arêtes à surcoût restent dans $E'$ mais leur poids effectif $\tilde{w}(i,j)$ est majoré, ce qui les défavorise naturellement dans la minimisation de l'objectif.

---
## 6. Formulation complète du modèle TSPTW-R <a id="6"></a>

Le modèle complet (**TSP with Time Windows and Edge Restrictions**, noté TSPTW-R) se formule comme un programme linéaire en nombres entiers mixtes (MILP) :

### Données

- $G = (V, E', \tilde{w})$ : graphe orienté pondéré avec arêtes admissibles seulement
- $n = |V|$ : nombre de sommets
- $[a_i, b_i]$ : fenêtre temporelle du sommet $i$
- $s_i$ : temps de service au sommet $i$
- $M$ : constante big-M

### Variables

- $x_{ij} \in \{0, 1\}$ : 1 si l'arête $(i,j)$ est utilisée
- $t_i \geq 0$ : temps d'arrivée au sommet $i$
- $u_i \in \{1, \ldots, n\}$ : ordre de visite du sommet $i$ (MTZ)

### Programme d'optimisation

$$\boxed{\min \sum_{(i,j) \in E'} \tilde{w}(i,j) \cdot x_{ij}}$$

Sous les contraintes :

$$\text{(C1a)} \quad \sum_{j \neq i} x_{ij} = 1 \qquad \forall i \in V \qquad \text{[chaque ville est quittée une fois]}$$

$$\text{(C1b)} \quad \sum_{i \neq j} x_{ij} = 1 \qquad \forall j \in V \qquad \text{[chaque ville est entrée une fois]}$$

$$\text{(C2)} \quad u_i - u_j + n \cdot x_{ij} \leq n-1 \qquad \forall i \neq j \in V \setminus \{v_0\} \qquad \text{[élimination des sous-tours]}$$

$$\text{(C3)} \quad t_j \geq t_i + s_i + \tilde{w}(i,j) - M(1 - x_{ij}) \qquad \forall (i,j) \in E' \qquad \text{[cohérence temporelle]}$$

$$\text{(C4)} \quad a_i \leq t_i \leq b_i \qquad \forall i \in V \setminus \{v_0\} \qquad \text{[fenêtres temporelles]}$$

$$\text{(C5)} \quad x_{ij} \in \{0, 1\}, \quad t_i \geq 0, \quad u_i \in \mathbb{Z}^+ \qquad \text{[domaines des variables]}$$

### Récapitulatif des contraintes

| Référence | Nature | Description |
|---|---|---|
| C1a, C1b | Flot | Chaque sommet est visité exactement une fois |
| C2 | Combinatoire | Élimination des sous-tours (MTZ) |
| C3 | Temporelle | Cohérence des temps de passage entre sommets consécutifs |
| C4 | **Time Windows** | Respect des créneaux horaires de chaque sommet |
| C5 | Intégrité | Domaines des variables de décision |

---
## 7. Analyse de la complexité théorique <a id="7"></a>

### 7.1 Rappels sur la théorie de la complexité

Un problème de décision appartient à la classe **NP** si, étant donné un certificat (une solution candidate), on peut vérifier en temps polynomial que ce certificat est valide. Un problème est dit **NP-difficile** si tout problème de NP se réduit polynomialement à lui. Un problème NP-difficile qui est lui-même dans NP est dit **NP-complet** [5].

### 7.2 Le TSP classique est NP-difficile

**Théorème (Karp, 1972 [6]) :** Le problème de décision associé au TSP — *"Existe-t-il une tournée hamiltonienne de coût ≤ K ?"* — est **NP-complet**.

**Preuve par réduction depuis le problème du cycle hamiltonien (HC) :**

Soit $G = (V, E)$ une instance de HC (problème NP-complet [6]). On construit une instance de TSP-décision :
- Graphe complet $G' = (V, V \times V)$
- Coûts : $w'(i,j) = 0$ si $(i,j) \in E$, $w'(i,j) = 1$ sinon
- Seuil : $K = 0$

Alors $G$ possède un cycle hamiltonien **si et seulement si** $G'$ admet une tournée de coût $\leq 0$. Cette construction est réalisable en temps polynomial $O(|V|^2)$. Donc TSP est NP-difficile. $\blacksquare$

### 7.3 Le TSPTW est NP-difficile

**Théorème :** Le TSP with Time Windows (TSPTW) est **NP-difficile**.

**Preuve :** Le TSP classique est un cas particulier du TSPTW où toutes les fenêtres temporelles sont $[0, +\infty[$. Formellement :

$$\text{TSP} \leq_P \text{TSPTW}$$

En effet, toute instance du TSP peut être transformée en instance du TSPTW en temps polynomial $O(|V|)$ en posant $a_i = 0$, $b_i = +\infty$, $s_i = 0$ pour tout $i$. Si TSPTW était soluble en temps polynomial, TSP le serait aussi — contradiction avec la NP-difficulté de TSP. Donc TSPTW est NP-difficile. $\blacksquare$

Ce résultat est établi par Savelsbergh (1985) [4] et confirme que l'ajout de fenêtres temporelles ne simplifie pas le problème.

### 7.4 Le TSPTW-R est NP-difficile

**Théorème :** Le TSPTW avec restrictions d'arêtes (TSPTW-R) est **NP-difficile**.

**Preuve :** Le TSPTW est un cas particulier du TSPTW-R où $c(i,j) = 1$ pour toutes les arêtes. Donc :

$$\text{TSPTW} \leq_P \text{TSPTW-R}$$

Par transitivité de la réductibilité polynomiale et NP-difficulté du TSPTW :

$$\text{TSP} \leq_P \text{TSPTW} \leq_P \text{TSPTW-R}$$

Le TSPTW-R est donc **NP-difficile**. $\blacksquare$

Les restrictions d'arêtes peuvent de surcroît rendre certaines instances **infaisables** (graphe non-hamiltonien), ajoutant une dimension combinatoire supplémentaire absente du TSP classique.

### 7.5 Conséquences pratiques

| Taille $n$ | Nombre de tournées possibles | Faisabilité d'un algorithme exhaustif |
|---|---|---|
| 5 | $4! = 24$ | Trivial |
| 10 | $9! = 362\,880$ | Rapide |
| 15 | $14! \approx 87 \times 10^9$ | Quelques minutes |
| 20 | $19! \approx 1.2 \times 10^{17}$ | Impossible en temps raisonnable |
| 50 | $49! \approx 10^{62}$ | Hors de portée |

La croissance factorielle du nombre de solutions justifie pleinement le recours à des **algorithmes exacts efficaces** (branch & bound, Held-Karp) et à des **métaheuristiques** pour les grandes instances.

### 7.6 Absence d'approximation polynomiale générale

Il est démontré que, sauf si P = NP, il n'existe **aucun algorithme d'approximation polynomial** avec ratio constant pour le TSP général (Sahni & Gonzalez, 1976 [7]). Cependant, pour le TSP métrique (inégalité triangulaire), l'algorithme de Christofides (1976) [8] garantit un ratio de $\frac{3}{2}$, et l'algorithme de Karlin, Klein & Gharan (2021) améliore légèrement ce ratio.

**Note :** La présence de fenêtres temporelles dures rend l'approximation encore plus difficile, car une solution approchée peut violer des contraintes de faisabilité.

---
## 8. Références bibliographiques <a id="8"></a>

[1] **Dantzig, G.B., Fulkerson, D.R., Johnson, S.M.** (1954). *Solution of a large-scale traveling-salesman problem*. Journal of the Operations Research Society of America, 2(4), 393-410. [DOI:10.1287/opre.2.4.393](https://doi.org/10.1287/opre.2.4.393)

[2] **Miller, C.E., Tucker, A.W., Zemlin, R.A.** (1960). *Integer programming formulation of traveling salesman problems*. Journal of the ACM, 7(4), 326-329. [DOI:10.1145/321043.321046](https://doi.org/10.1145/321043.321046)

[3] **Desrochers, M., Laporte, G.** (1991). *Improvements and extensions to the Miller-Tucker-Zemlin subtour elimination constraints*. Operations Research Letters, 10(1), 27-36. [DOI:10.1016/0167-6377(91)90083-2](https://doi.org/10.1016/0167-6377(91)90083-2)

[4] **Savelsbergh, M.W.P.** (1985). *Local search in routing problems with time windows*. Annals of Operations Research, 4(1), 285-305. [DOI:10.1007/BF02022044](https://doi.org/10.1007/BF02022044)

[5] **Garey, M.R., Johnson, D.S.** (1979). *Computers and Intractability: A Guide to the Theory of NP-Completeness*. W.H. Freeman and Company. ISBN: 978-0-7167-1045-5.

[6] **Karp, R.M.** (1972). *Reducibility among combinatorial problems*. In R.E. Miller & J.W. Thatcher (Eds.), *Complexity of Computer Computations* (pp. 85-103). Plenum Press. [DOI:10.1007/978-1-4684-2001-2_9](https://doi.org/10.1007/978-1-4684-2001-2_9)

[7] **Sahni, S., Gonzalez, T.** (1976). *P-complete approximation problems*. Journal of the ACM, 23(3), 555-565. [DOI:10.1145/321958.321975](https://doi.org/10.1145/321958.321975)

[8] **Christofides, N.** (1976). *Worst-case analysis of a new heuristic for the travelling salesman problem*. Technical Report 388, Graduate School of Industrial Administration, Carnegie-Mellon University.

[9] **Solomon, M.M.** (1987). *Algorithms for the vehicle routing and scheduling problems with time window constraints*. Operations Research, 35(2), 254-265. [DOI:10.1287/opre.35.2.254](https://doi.org/10.1287/opre.35.2.254)

[10] **Toth, P., Vigo, D.** (Eds.) (2002). *The Vehicle Routing Problem*. SIAM Monographs on Discrete Mathematics and Applications. ISBN: 978-0-89871-498-2.

---

## Conclusion du livrable

Ce notebook a présenté une modélisation complète du problème TSPTW-R (**TSP with Time Windows and Edge Restrictions**), version enrichie du TSP classique retenue pour ce projet.

Les éléments couverts sont :

- **Contexte applicatif** : enjeux ADEME, cas d'usage de livraison
- **Représentation formelle** : graphe orienté $G = (V, E, w, c)$, matrices $W$, $C$, $\tilde{W}$
- **Formulation MILP** : variables $x_{ij}$, $t_i$, $u_i$ — objectif et 5 familles de contraintes
- **Contraintes supplémentaires** : fenêtres temporelles (C4, C3) et restrictions d'arêtes ($c$ dans $\tilde{w}$)
- **Preuve de NP-difficulté** : par deux réductions polynomiales enchaînées HC → TSP → TSPTW → TSPTW-R
- **Implémentation** : structures de données, vérification de faisabilité, validation de solutions

Les méthodes de résolution (algorithme exact et métaheuristique) seront développées dans les livrables suivants.